In [13]:
import requests
import pandas as pd
from io import StringIO

In [14]:
# -------------------------------------------------------
# 0) Proxy settings (from your FoxyProxy config)
# -------------------------------------------------------
proxies = {
    "http":  "http://localhost:3128",
    "https": "http://localhost:3128",
}

session = requests.Session()
session.proxies.update(proxies)

In [15]:
import browser_cookie3

# If you use Firefox:
session.cookies.update(browser_cookie3.firefox(domain_name="sdcc.bnl.gov"))
session.cookies.update(browser_cookie3.firefox(domain_name="phenix-intra.sdcc.bnl.gov"))



In [16]:
import re
import pandas as pd
from io import StringIO

def _flatten_cols(df):
    # pandas sometimes returns MultiIndex columns from HTML tables
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join([str(x) for x in tup if str(x) != "nan"]).strip() for tup in df.columns]
    else:
        df.columns = [str(c).strip() for c in df.columns]
    return df

def _canon(s):
    # canonical key: ignore spaces/underscores/case
    return re.sub(r"[\s_]+", "", str(s)).lower()

def fetch_trigger_table(run, timeout=30):
    """
    Fetch RunSummary trigger table and return a DataFrame with at least:
      Name, Live Trigger Count, Scaled Trigger Count
    """
    url = f"https://phenix-intra.sdcc.bnl.gov/WWW/run/daq/runcontrol/RunSummary.php?RunNumber={run}"

    r = session.get(url, allow_redirects=False, timeout=timeout)

    # If you’re not authenticated you get redirected to Keycloak
    if r.status_code in (301, 302, 303, 307, 308):
        loc = r.headers.get("Location", "")
        if "auth.sdcc.bnl.gov" in loc:
            raise RuntimeError("SSO redirect detected. Attach SDCC cookies to `session` first.")

    if r.status_code in (401, 403):
        raise RuntimeError(f"Unauthorized ({r.status_code}). Attach SDCC cookies to `session` first.")

    r.raise_for_status()

    tables = pd.read_html(StringIO(r.text))
    for df in tables:
        df = _flatten_cols(df)

        # Identify the trigger table by key columns
        cols = [_canon(c) for c in df.columns]
        if "name" in cols and any("livetriggercount" in c for c in cols):
            # normalize Name
            name_col = df.columns[cols.index("name")]
            df[name_col] = df[name_col].astype(str).str.strip()
            df = df.rename(columns={name_col: "Name"})
            return df

    raise RuntimeError("Could not find trigger table in HTML (SSO page or format changed).")


In [17]:
def get_selected_trigger_counts(run, trigger_names):
    df = fetch_trigger_table(run)

    # Find the exact column names for counts (HTML changes sometimes)
    colmap = { _canon(c): c for c in df.columns }

    live_col   = None
    scaled_col = None

    for k, orig in colmap.items():
        if "livetriggercount" in k:
            live_col = orig
        if "scaledtriggercount" in k:
            scaled_col = orig

    if live_col is None:
        raise RuntimeError(f"Live Trigger Count column not found. Columns: {list(df.columns)}")
    if scaled_col is None:
        raise RuntimeError(f"Scaled Trigger Count column not found. Columns: {list(df.columns)}")

    # Build lookup by canonicalized name
    df["_key"] = df["Name"].map(_canon)
    lut = df.set_index("_key")[["Name", live_col, scaled_col]]

    rows = []
    for req in trigger_names:
        key = _canon(req)
        if key in lut.index:
            row = lut.loc[key]
            rows.append({
                "Requested": req,
                "MatchedName": row["Name"],
                "LiveTriggerCount": int(row[live_col]) if pd.notna(row[live_col]) else None,
                "ScaledTriggerCount": int(row[scaled_col]) if pd.notna(row[scaled_col]) else None,
            })
        else:
            rows.append({
                "Requested": req,
                "MatchedName": None,
                "LiveTriggerCount": None,
                "ScaledTriggerCount": None,
            })

    return pd.DataFrame(rows)


In [18]:
TRIGGERS = [
    "CLOCK",
    "ERTLL1_E&BBCLL1(narrow)",
    "ERT_4x4c&BBCLL1",
    "ERT_4x4a&BBCLL1",
    "ERT 4x4b",                    # note: table shows a space in your screenshot
    "BBCLL1(>0 tubes)",
    "BBCLL1(>0 tubes) narrowvtx",
]

df_sel = get_selected_trigger_counts(421815, TRIGGERS)
df_sel


,Requested,MatchedName,LiveTriggerCount,ScaledTriggerCount
0,CLOCK,CLOCK,34223700425,174541
1,ERTLL1_E&BBCLL1(narrow),ERTLL1_E&BBCLL1(narrow),793413,793413
2,ERT_4x4c&BBCLL1,ERT_4x4c&BBCLL1,12996752,12996752
3,ERT_4x4a&BBCLL1,ERT_4x4a&BBCLL1,2031280,2031280
4,ERT 4x4b,ERT_4x4b,368109,368109
5,BBCLL1(>0 tubes),BBCLL1(>0 tubes),1123400375,255318
6,BBCLL1(>0 tubes) narrowvtx,BBCLL1(>0 tubes) narrowvtx,440156855,222301


In [22]:
import pandas as pd

def get_trigger_counts_for_runs(runs, trigger_names, save_csv=None, verbose=True):
    """
    Loop over runs and return a tidy table of selected trigger counts.

    Parameters
    ----------
    runs : iterable of int
        Run numbers.
    trigger_names : list[str]
        Trigger names to extract (as shown on RunSummary table).
    save_csv : str or None
        If given, write results to this CSV path.
    verbose : bool
        Print progress and errors.

    Returns
    -------
    pandas.DataFrame with columns:
      Run, Requested, MatchedName, LiveTriggerCount, ScaledTriggerCount
    """
    out_rows = []

    runs = list(runs)
    for i, run in enumerate(runs, start=1):
        try:
            if verbose:
                print(f"[{i}/{len(runs)}] run={run}")

            df_one = get_selected_trigger_counts(run, trigger_names)  # uses your earlier function
            df_one.insert(0, "Run", int(run))
            out_rows.append(df_one)

        except Exception as e:
            if verbose:
                print(f"  ❌ run={run} failed: {e}")

            # still record rows for this run so output stays rectangular
            for t in trigger_names:
                out_rows.append({
                    "Run": int(run),
                    "Requested": t,
                    "MatchedName": None,
                    "LiveTriggerCount": None,
                    "ScaledTriggerCount": None,
                })

    df_all = pd.concat([pd.DataFrame(r) if not isinstance(r, pd.DataFrame) else r for r in out_rows],
                       ignore_index=True)

    # optional: compute ratio to CLOCK live count per run (handy)
    # Adds a column LiveOverClock for non-CLOCK triggers.
    if "CLOCK" in [t.upper() for t in trigger_names]:
        # build per-run clock live
        clock_key = "clock"
        tmp = df_all.copy()
        tmp["_reqkey"] = tmp["Requested"].map(lambda x: _canon(x))
        clock_live = (tmp[tmp["_reqkey"] == clock_key]
                      .groupby("Run")["LiveTriggerCount"]
                      .first()
                      .rename("ClockLive"))
        df_all = df_all.merge(clock_live, on="Run", how="left")
        df_all["LiveOverClock"] = df_all["LiveTriggerCount"] / df_all["ClockLive"]

        df_all = df_all.drop(columns=["ClockLive"], errors="ignore")

    if save_csv:
        df_all.to_csv(save_csv, index=False)
        if verbose:
            print(f"✅ wrote {save_csv}")

    return df_all


In [23]:


# Multiple runs
runs = [421815, 421816, 421949]   # replace with your run list
df_runs = get_trigger_counts_for_runs(runs, TRIGGERS)
df_runs


[1/3] run=421815
[2/3] run=421816
[3/3] run=421949


,Run,Requested,MatchedName,LiveTriggerCount,ScaledTriggerCount,LiveOverClock
0,421815,CLOCK,CLOCK,34223700425,174541,1.000000
1,421815,ERTLL1_E&BBCLL1(narrow),ERTLL1_E&BBCLL1(narrow),793413,793413,0.000023
2,421815,ERT_4x4c&BBCLL1,ERT_4x4c&BBCLL1,12996752,12996752,0.000380
3,421815,ERT_4x4a&BBCLL1,ERT_4x4a&BBCLL1,2031280,2031280,0.000059
4,421815,ERT 4x4b,ERT_4x4b,368109,368109,0.000011
5,421815,BBCLL1(>0 tubes),BBCLL1(>0 tubes),1123400375,255318,0.032825
6,421815,BBCLL1(>0 tubes) narrowvtx,BBCLL1(>0 tubes) narrowvtx,440156855,222301,0.012861
7,421816,CLOCK,CLOCK,29281025673,149333,1.000000
8,421816,ERTLL1_E&BBCLL1(narrow),ERTLL1_E&BBCLL1(narrow),3265877,3265877,0.000112
9,421816,ERT_4x4c&BBCLL1,ERT_4x4c&BBCLL1,8776542,4388271,0.000300


In [24]:
runs = [421815, 421816, 421949, 421961, 421968, 421969, 421975, 421976, 421988, 421989, 421999, 422014, 422018, 422020, 422040, 422041, 422043, 422045, 422050, 422051, 422053, 422054, 422055, 422057, 422064, 422066, 422067, 422068, 422070, 422074, 422075, 422084, 422085, 422123, 422124, 422135, 422141, 422147, 422148, 422200, 422201, 422202, 422203, 422205, 422255, 422256, 422260, 422262, 422263, 422266, 422267, 422268, 422269, 422272, 422273, 422298, 422305, 422314, 422319, 422322, 422323, 422324, 422531, 422540, 422543, 422553, 422562, 422566, 422575, 422611, 422613, 422614, 422615, 422618, 422633, 422640, 422643, 422756, 422758, 422762, 422763, 422779, 422782, 422785, 423041, 423043, 423101, 423109, 423110, 423132, 423149, 423220, 423263, 423268, 423275, 423278, 423290, 423301, 423311, 423377, 423424, 423432, 423433, 423546, 423547, 423548, 423549, 423550, 423553, 423576, 423579, 423632, 423652, 423663, 423664, 423676, 423677, 423679, 423685, 423696, 423703, 423820, 423826, 423828, 423839, 423843, 423844, 423845, 423856, 423860, 423865, 423867, 423868, 424038, 424039, 424040, 424051, 424052, 424053, 424055, 424060, 424061, 424062, 424063, 424189, 424190, 424193, 424194, 424195, 424227, 424255, 424257, 424270, 424272, 424273, 424275, 424276, 424348, 424353, 424356, 424358, 424365, 424368, 424371, 424372, 424439, 424440, 424441, 424443, 424444, 424445, 424452, 424557, 424584, 424585, 424626, 424627, 424629, 424644, 424671, 424751, 424754, 424755, 424756, 424759, 424760, 424761, 424763, 424817, 424827, 424836, 424877, 424880, 424881, 424885, 424886, 425006, 425008, 425011, 425016, 425078, 425079, 425080, 425152, 425168, 425171, 425172, 425173, 425212, 425213, 425214, 425215, 425286, 425287, 425288, 425289, 425290, 425292, 425293, 425294, 425295, 425296, 425377, 425378, 425395, 425397, 425403, 425409, 425412, 425413, 425420, 425422, 425423, 425424, 425425, 425428, 425429, 425431, 425433, 425434, 425439, 425553, 425564, 425566, 425583, 425688, 425691, 425692, 425693, 425694, 426112, 426113, 426114, 426115, 426116, 426117, 426252, 426273, 426281, 426282, 426283, 426307, 426308, 426313, 426315, 426316, 426319, 426320, 426353, 426365, 426366, 426378, 426383, 426391, 426394, 426397, 426401, 426406, 426407, 426408, 426409, 426442, 426443, 426444, 426445, 426450, 426460, 426461, 427013, 427019, 427020, 427021, 427024, 427025, 427026, 427027, 427111, 427129, 427135, 427136, 427137, 427141, 427144, 427145, 427146, 427147, 427148, 427149, 427233, 427239, 427241, 427243, 427244, 427262, 427264, 427360, 427361, 427362, 427363, 427366, 427367, 427373, 427377, 427380, 427383, 427384, 427385, 427388, 427389, 427390, 427391, 427393, 427394, 427397, 427398, 427481, 427482, 427483, 427484, 427485, 427486, 427498, 427499, 427500, 427502, 427508, 427510, 427514, 427523, 427527, 427529, 427530, 427605, 427654, 427656, 427657, 427658, 427660, 427661, 427662, 427670, 427671, 427672, 427673, 427674, 427708, 427709, 427710, 427711, 427712, 427713, 427805, 427806, 427807, 427810, 427811, 427813, 427814, 427815, 427829, 427878, 427879, 427881, 427882, 427883, 427885, 427886, 427887, 427963, 427964, 427965, 427968, 427970, 427974, 427975, 427977, 427979, 427980, 427982, 427984, 428166, 428168, 428169, 428171, 428204, 428206, 428207, 428208, 428211, 428212, 428255, 428256, 428260, 428261, 428262, 428263, 428264, 428265, 428266, 428267, 428268, 428269, 428272, 428273, 428318, 428319, 428321, 428323, 428324, 428325, 428326, 428328, 428329, 428331, 428377, 428379, 428381, 428382, 428384, 428385, 428386, 428431, 428432, 428433, 428446, 428447, 428448, 428451, 428452, 428453, 428454, 428455, 428459, 428460, 428601, 428602, 428603, 428604, 428605, 428607, 428608, 428609, 428610, 428613, 428614, 428615, 428616, 428617, 428618, 428710, 428713, 428714, 428715, 428717, 428728, 428730, 428733, 428734, 428735, 428736, 428737, 428738, 428739, 428754, 428755, 428757, 428758, 428759, 428760, 428762, 428763, 428771, 428772, 428891, 428892, 428893, 428894, 428896, 428898, 428929, 428930, 428931, 428933, 428934, 429007, 429010, 429014, 429016, 429017, 429022, 429023, 429024, 429025, 429026, 429027, 429029, 429062, 429066, 429067, 429068, 429069, 429070, 429071, 429112, 429114, 429115, 429126, 429127, 429128, 429129, 429132, 429133, 429351, 429352, 429353, 429355, 429359, 429361, 429364, 429365, 429366, 429368, 429370, 429503, 429504, 429505, 429506, 429512, 429518, 429519, 429549, 429551, 429552, 429554, 429555, 429556, 429589, 429590, 429591, 429592, 429593, 429594, 429595, 429596, 429676, 429678, 429679, 429680, 429685, 429686, 429687, 429688, 429689, 429691, 429696, 429787, 429789, 429795, 429796, 429797, 429798, 429799, 429800, 429801, 429802, 429886, 429887, 429888, 429889, 429890, 429893, 429894, 429895, 429896, 429905, 429906, 429909, 429910, 429911, 429912, 429915, 430013, 430014, 430016, 430017, 430022, 430023, 430024, 430116, 430117, 430119, 430120, 430121, 430123, 430124, 430125, 430128, 430131, 430133, 430134, 430136, 430141, 430142, 430143, 430234, 430235, 430236, 430237, 430238, 430239, 430240, 430241, 430242, 430277, 430278, 430279, 430280, 430281, 430384, 430386, 430389, 430390, 430393, 430402, 430406, 430407, 430408, 430409, 430414, 430415, 430494, 430496, 430497, 430500, 430501, 430502, 430519, 430520, 430521, 430522, 430524, 430525, 430557, 430558, 430560, 430562, 430563, 430565, 430566, 430594, 430595, 430596, 430598, 430599, 430600, 430607, 430608, 430676, 430679, 430680, 430681, 430682, 430683, 430692, 430693, 430694, 430696, 430697, 430699, 430700, 430701, 430702, 430905, 430906, 430907, 430909, 430911, 430912, 430913, 430914, 430920, 430921, 430923, 430924, 430925, 430927, 430928, 430929, 430930, 430931, 430932, 430933, 430935, 430936, 431020, 431021, 431022, 431023, 431026, 431027, 431028, 431030, 431031, 431033, 431040, 431122, 431123, 431125, 431126, 431127, 431130, 431131, 431134, 431135, 431136, 431137, 431138, 431139, 431142, 431143, 431144, 431145, 431146, 431147, 431148, 431149, 431216, 431217, 431219, 431220, 431221, 431224, 431233, 431234, 431235, 431239, 431240, 431256, 431257, 431258, 431259, 431260, 431261, 431294, 431295, 431298, 431299, 431301, 431302, 431357, 431358, 431360, 431361, 431362, 431428, 431429, 431430, 431432, 431437, 431447, 431448, 431453, 431454, 431458, 431608, 431609, 431612, 431615, 431616, 431618, 431619, 431620, 431622, 431717, 431720, 431725, 431727, 431731, 431732, 431733, 431736, 431738, 431739, 431744, 431745, 431746, 431831, 431832, 431833, 431834, 431835, 431836, 431837, 431839, 431840, 431844, 431845, 431846, 431854, 431859, 431860, 431886, 431888, 431889, 431891, 431892, 431893, 431894, 431932, 431937, 431938, 431939, 431940, 431941, 431942, 431943, 431948, 431951, 431960, 431963, 431997, 432001, 432007, 432008]


In [ ]:

if True:
    df_runs = get_trigger_counts_for_runs(runs, TRIGGERS)
    df_runs.to_csv("output/phenix_trigger_live_counts.csv", index=False)
    df_runs

[1/844] run=421815
[2/844] run=421816
[3/844] run=421949
[4/844] run=421961
[5/844] run=421968
[6/844] run=421969
[7/844] run=421975
[8/844] run=421976
[9/844] run=421988
[10/844] run=421989
[11/844] run=421999
[12/844] run=422014
[13/844] run=422018
[14/844] run=422020
[15/844] run=422040
[16/844] run=422041
[17/844] run=422043
[18/844] run=422045
[19/844] run=422050
[20/844] run=422051
[21/844] run=422053
[22/844] run=422054
[23/844] run=422055
[24/844] run=422057
[25/844] run=422064
[26/844] run=422066
[27/844] run=422067
[28/844] run=422068
[29/844] run=422070
[30/844] run=422074
[31/844] run=422075
[32/844] run=422084
[33/844] run=422085
[34/844] run=422123
[35/844] run=422124
[36/844] run=422135
[37/844] run=422141
[38/844] run=422147
[39/844] run=422148
[40/844] run=422200
[41/844] run=422201
[42/844] run=422202
[43/844] run=422203
[44/844] run=422205
[45/844] run=422255
[46/844] run=422256
[47/844] run=422260
[48/844] run=422262
[49/844] run=422263
[50/844] run=422266
[51/844] 

,Run,Requested,MatchedName,LiveTriggerCount,ScaledTriggerCount,LiveOverClock
0,421815,CLOCK,CLOCK,3.422370e+10,174541.0,1.000000
1,421815,ERTLL1_E&BBCLL1(narrow),ERTLL1_E&BBCLL1(narrow),7.934130e+05,793413.0,0.000023
2,421815,ERT_4x4c&BBCLL1,ERT_4x4c&BBCLL1,1.299675e+07,12996752.0,0.000380
3,421815,ERT_4x4a&BBCLL1,ERT_4x4a&BBCLL1,2.031280e+06,2031280.0,0.000059
4,421815,ERT 4x4b,ERT_4x4b,3.681090e+05,368109.0,0.000011
...,...,...,...,...,...,...
5903,432008,ERT_4x4c&BBCLL1,None,NaN,NaN,NaN
5904,432008,ERT_4x4a&BBCLL1,ERT_4x4a&BBCLL1,3.846793e+06,3846793.0,0.000118
5905,432008,ERT 4x4b,ERT_4x4b,3.648080e+05,364808.0,0.000011
5906,432008,BBCLL1(>0 tubes),BBCLL1(>0 tubes),1.767398e+09,357050.0,0.054418


In [27]:
df_runs = pd.read_csv("output/phenix_trigger_live_counts.csv")
df_runs

,Run,Requested,MatchedName,LiveTriggerCount,ScaledTriggerCount,LiveOverClock
0,421815,CLOCK,CLOCK,3.422370e+10,174541.0,1.000000
1,421815,ERTLL1_E&BBCLL1(narrow),ERTLL1_E&BBCLL1(narrow),7.934130e+05,793413.0,0.000023
2,421815,ERT_4x4c&BBCLL1,ERT_4x4c&BBCLL1,1.299675e+07,12996752.0,0.000380
3,421815,ERT_4x4a&BBCLL1,ERT_4x4a&BBCLL1,2.031280e+06,2031280.0,0.000059
4,421815,ERT 4x4b,ERT_4x4b,3.681090e+05,368109.0,0.000011
...,...,...,...,...,...,...
5903,432008,ERT_4x4c&BBCLL1,NaN,NaN,NaN,NaN
5904,432008,ERT_4x4a&BBCLL1,ERT_4x4a&BBCLL1,3.846793e+06,3846793.0,0.000118
5905,432008,ERT 4x4b,ERT_4x4b,3.648080e+05,364808.0,0.000011
5906,432008,BBCLL1(>0 tubes),BBCLL1(>0 tubes),1.767398e+09,357050.0,0.054418


In [29]:
import numpy as np
import pandas as pd
import re

def _canon(s):
    return re.sub(r"[\s_]+", "", str(s)).lower()

def calc_mb_from_ert(df_runs,
                     mb_trigger="BBCLL1(>0 tubes) narrowvtx",
                     ert_trigger="ERT_4x4a&BBCLL1"):
    """
    Uses df_runs from get_trigger_counts_for_runs(...)

    Returns:
      summary dict with totals + ratio,
      and a per-run dataframe with MB_raw, ERT_scaled, ratio.
    """
    # sanity check columns
    need = {"Run", "Requested", "LiveTriggerCount", "ScaledTriggerCount"}
    missing = need - set(df_runs.columns)
    if missing:
        raise RuntimeError(f"df_runs missing columns: {missing}. "
                           f"Available: {list(df_runs.columns)}")

    # build wide table by Run x Requested for Raw and Scaled
    df = df_runs.copy()
    df["reqkey"] = df["Requested"].map(_canon)

    wide_raw = df.pivot_table(index="Run", columns="reqkey",
                              values="LiveTriggerCount", aggfunc="first")
    wide_scl = df.pivot_table(index="Run", columns="reqkey",
                              values="ScaledTriggerCount", aggfunc="first")

    mb_key  = _canon(mb_trigger)
    ert_key = _canon(ert_trigger)

    if mb_key not in wide_raw.columns:
        raise RuntimeError(f"MB trigger '{mb_trigger}' not found in LiveTriggerCount. "
                           f"Have: {list(wide_raw.columns)}")
    if ert_key not in wide_scl.columns:
        raise RuntimeError(f"ERT trigger '{ert_trigger}' not found in ScaledTriggerCount. "
                           f"Have: {list(wide_scl.columns)}")

    per_run = pd.DataFrame({
        "Run": wide_raw.index.astype(int),
        "MB_raw": wide_raw[mb_key].astype("float64"),
        "ERT_scaled": wide_scl[ert_key].astype("float64"),
    }).reset_index(drop=True)

    # keep only good rows
    per_run = per_run[(per_run["MB_raw"].notna()) &
                      (per_run["ERT_scaled"].notna()) &
                      (per_run["ERT_scaled"] > 0)]

    per_run["MB_over_ERT"] = per_run["MB_raw"] / per_run["ERT_scaled"]

    MB_raw_total = float(per_run["MB_raw"].sum())
    ERT_scaled_total = float(per_run["ERT_scaled"].sum())
    ratio_total = MB_raw_total / ERT_scaled_total if ERT_scaled_total > 0 else np.nan

    summary = {
        "mb_trigger": mb_trigger,
        "ert_trigger": ert_trigger,
        "MB_raw_total": MB_raw_total,
        "ERT_scaled_total": ERT_scaled_total,
        "ratio_MB_over_ERT_total": ratio_total
    }
    return summary, per_run

# ---- usage ----
summary, per_run = calc_mb_from_ert(
    df_runs,
    mb_trigger="BBCLL1(>0 tubes) narrowvtx",  # <-- narrow MB
    ert_trigger="ERT_4x4a&BBCLL1"             # <-- your ERT trigger
)

summary, per_run.head()


({'mb_trigger': 'BBCLL1(>0 tubes) narrowvtx',
  'ert_trigger': 'ERT_4x4a&BBCLL1',
  'MB_raw_total': 502352398131.0,
  'ERT_scaled_total': 1346232941.0,
  'ratio_MB_over_ERT_total': 373.15414207428756},
       Run       MB_raw  ERT_scaled  MB_over_ERT
 0  421815  440156855.0   2031280.0   216.689405
 1  421816  320114454.0   1423949.0   224.807528
 2  421949  552356396.0   1680045.0   328.774763
 3  421961  408388788.0   1315109.0   310.536076
 4  421968  428615594.0   1573420.0   272.410160)

In [32]:
print(f"MB_raw_total: {summary['MB_raw_total']:.0f}, ERT_scaled_total: {summary['ERT_scaled_total']:.0f}, ratio_MB_over_ERT_total: {summary['ratio_MB_over_ERT_total']:.6f}")

MB_raw_total: 502352398131, ERT_scaled_total: 1346232941, ratio_MB_over_ERT_total: 373.154142


In [35]:
my_n_ev = 1.07272 * 10**9
print(summary['MB_raw_total'] / summary['ERT_scaled_total'] * my_n_ev / summary['ERT_scaled_total']  )

297.34074920837173
